In [1]:
import os
import sys
import shutil
from google.cloud import storage
from tqdm import tqdm

In [2]:
storage_client = storage.Client.from_service_account_json('creds.json')
bucket = storage_client.get_bucket("rheyminecraft-mods") 

In [3]:
minecraft_dir = "C:\\Users\\Rhey PC\\AppData\\Roaming\\.minecraft"

In [5]:
# Function for GOOGLE CLOUD STORAGE!
def get_all_blobs():
    """
    Returns a list of all blob names in the bucket.
    """
    filename=list(bucket.list_blobs())
    files = [blob.name for blob in filename]
    return files

def rename_blobs(original_blob, renamed_blob):
    """
    Renames a blob in the bucket.
    
    Parameters
    ------------
    - original_blob (str):
        The name of the blob to rename.
    - renamed_blob (str):
        The new name for the blob.
    """
    if original_blob in get_all_blobs():
        blob = bucket.blob(original_blob)
        new_blob = bucket.rename_blob(blob, renamed_blob)
        print(f"Blob {original_blob} renamed to {renamed_blob}")
    else:
        print(f"Blob {original_blob} does not exist.")

def rename_blob_folder(original_folder, renamed_folder):
    """
    Renames all blobs in a folder in the bucket.

    Parameters
    ------------
    - original_folder (str):
        The name of the folder containing blobs to rename.
    - renamed_folder (str):
        The new name for the folder.
    """
    original_folder = f'{original_folder}/'
    renamed_folder = f'{renamed_folder}/'
    blobs = [i for i in get_all_blobs() if i.startswith(original_folder)]
    for old_blob in blobs:
        new_blob = old_blob.replace(original_folder, renamed_folder)
        old_blob = bucket.blob(old_blob)
        a = bucket.rename_blob(old_blob, new_blob)
    print("Rename success!")

def upload_to_bucket(blob_name, local_path):
    """
    Uploads a file to the bucket.

    Parameters
    ------------
    - blob_name (str):
        The name of the blob in the bucket.
    - local_path (str):
        The local path to the file to upload.
    """
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(local_path)


def datapacks():
    '''
    Keeps the datapacks folder in sync with the GCS bucket.
    '''
    tacz_dir = os.path.join(minecraft_dir, "tacz")
    zip_files = [f for f in os.listdir(tacz_dir) if f.endswith('.zip')]
    gcs_files = [blob for blob in get_all_blobs() if blob.startswith('tacz/')]
    gcs_zip_files = [os.path.basename(f) for f in gcs_files]

    # Upload new or updated zip files to GCS
    for zip_file in zip_files:
        local_path = os.path.join(tacz_dir, zip_file)
        if zip_file not in gcs_zip_files:
            upload_to_bucket(f'tacz/{zip_file}', local_path)
            print(f'Uploaded new file: {zip_file}')

    # Delete GCS files that are no longer present locally
    for gcs_file in gcs_zip_files:
        if gcs_file not in zip_files:
            blob = bucket.blob(f'tacz/{gcs_file}')
            blob.delete()
            print(f'Deleted file from GCS: {gcs_file}')

    print('Datapacks sync complete.')


def mods():
    '''
    Keeps the mods folder in sync with the GCS bucket.
    '''
    mods_dir = os.path.join(minecraft_dir, "mods")
    jar_files = [f for f in os.listdir(mods_dir) if f.endswith('.jar')]
    gcs_files = [blob for blob in get_all_blobs() if blob.startswith('mods/')]
    gcs_jar_files = [os.path.basename(f) for f in gcs_files]

    # Upload new or updated jar files to GCS
    for jar_file in jar_files:
        local_path = os.path.join(mods_dir, jar_file)
        if jar_file not in gcs_jar_files:
            upload_to_bucket(f'mods/{jar_file}', local_path)
            print(f'Uploaded new file: {jar_file}')

    # Delete GCS files that are no longer present locally
    for gcs_file in gcs_jar_files:
        if gcs_file not in jar_files:
            blob = bucket.blob(f'mods/{gcs_file}')
            blob.delete()
            print(f'Deleted file from GCS: {gcs_file}')

    print('Mods sync complete.')

## Mods

In [6]:
mods()

Uploaded new file: tl_skin_cape_forge_1.20-1.36.jar
Uploaded new file: tl_skin_cape_forge_1.20.1-1.32.jar
Deleted file from GCS: MCSP-1.20.1-V1.0.5.jar
Deleted file from GCS: ashvehicle-4.1-8.7-SNAPSHOT.jar
Deleted file from GCS: superbwarfare-1.20.1-0.8.7-final-d6ea9a72b.jar
Deleted file from GCS: vvp-0.1.7.jar
Mods sync complete.


## Datapacks

In [42]:
datapacks()

Uploaded new file: 82laffey1.1.2(1).zip
Datapacks sync complete.


## Options

In [25]:
blob = bucket.blob("options.txt")
blob.upload_from_filename( os.path.join(minecraft_dir, "options.txt") )